# HF audio bytes walkthrough

Mirrors the image-bytes walkthrough in `examples/hf_dataset_test.ipynb`, but for audio.
Goal: show that **raw WAV bytes** live in parquet, and `datasets.Audio(decode=False)`
yields them straight to Python with no soundfile/librosa decode in between.

Dataset: [`rfcx/frugalai`](https://huggingface.co/datasets/rfcx/frugalai) — binary
classification (`chainsaw` vs `environment`), 3-second clips at 12 kHz.

In [ ]:
import datasets

ds = datasets.load_dataset("rfcx/frugalai", split="test")
ds

In [ ]:
ds.features

### Default: `Audio(decode=True)` — eager decode to a NumPy array

Indexing the dataset triggers `soundfile` to decode the WAV bytes from parquet
into a `float32` waveform on the way out.

In [ ]:
row = ds[0]
print("keys:", list(row.keys()))
print("audio:", {k: (v.shape if hasattr(v, 'shape') else v) for k, v in row['audio'].items()})
print("label:", row['label'], '->', ds.features['label'].int2str(row['label']))

### `Audio(decode=False)` — raw WAV bytes from parquet

Re-cast the column with `decode=False` and the dataset yields
`{"bytes": b"RIFF...", "path": "...wav"}` instead of the decoded array. The
bytes are the **original WAV file** sitting in the parquet — no decode happened.

In [ ]:
ds_raw = ds.cast_column("audio", datasets.Audio(decode=False))
raw = ds_raw[0]['audio']

print(f"type:        {type(raw).__name__}")
print(f"path:        {raw['path']}")
print(f"bytes len:   {len(raw['bytes'])}")
print(f"magic:       {raw['bytes'][:4]!r}    (RIFF -> WAV)")
print(f"WAVE marker: {raw['bytes'][8:12]!r}")

### Why it matters

When 3LC builds a Table from an HF audio dataset, the naive path is:

```
parquet WAV bytes → soundfile decode → numpy array
                                      → soundfile encode → on-disk WAV file
```

The bytes-fastpath skips both decode steps and writes the parquet bytes
**verbatim** to disk. It's the same trick already used for `datasets.Image`
in `_TableFromHuggingFaceBase`. The next notebook builds a 3LC table on top
of this fastpath.